# 01 — Data Understanding & Initial EDA

**Project:** VitaMap — ResumeForge  
**Branch:** `khushi-backend`  
**Phase:** 1 — Data Discovery → Understanding → Data Quality → Initial EDA  

**Rules enforced in this notebook:**
- Raw data (`backend/data/raw/`) is never modified.
- No learned transformations (vectorizer, scaler, encoder) fitted.
- No train/validation/test split.
- No model training.
- Technical tokens preserved (Python, SQL, C++, etc.).
- `random_state=42` for reproducibility.
- All analysis on working copy `df` — `df_raw` remains unmodified.

## 0. Imports & Paths

In [ ]:
import os
import re
import sys
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings('ignore')
random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR  = NOTEBOOK_DIR.parent
RAW_DIR      = BACKEND_DIR / 'data' / 'raw'
RAW_DATA_DIR = RAW_DIR / 'data'
EDA_DIR      = BACKEND_DIR / 'data' / 'processed' / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Backend  : {BACKEND_DIR}')
print(f'Raw dir  : {RAW_DIR}')
print(f'EDA out  : {EDA_DIR}')

---
## 1. Dataset Discovery

In [ ]:
# ── Source 1: Structured dataset ─────────────────────────────────────────────
SUPPORTED_EXTS = {'.csv', '.xlsx', '.xls', '.json', '.parquet'}
structured_files = [f for f in RAW_DIR.iterdir()
                    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS]

print('Structured files found in raw/:')
for f in structured_files:
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)  suffix={f.suffix}')

if not structured_files:
    raise FileNotFoundError('No structured dataset found in backend/data/raw/')

DATASET_FILE = structured_files[0]
print(f'\n→ Structured dataset: {DATASET_FILE.name}')

# ── Source 2: Raw resume folder ───────────────────────────────────────────────
print(f'\nRaw resume folder: {RAW_DATA_DIR}')
category_dirs = sorted([d for d in RAW_DATA_DIR.iterdir() if d.is_dir()])
print(f'Category folders found: {len(category_dirs)}')
for d in category_dirs:
    print(f'  {d.name}')

---
## 2. Load & Inspect Structured Dataset

In [ ]:
def load_dataset(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext == '.csv':
        return pd.read_csv(path, encoding='utf-8', encoding_errors='replace')
    elif ext in ('.xlsx', '.xls'):
        return pd.read_excel(path)
    elif ext == '.json':
        return pd.read_json(path)
    elif ext == '.parquet':
        return pd.read_parquet(path)
    else:
        raise ValueError(f'Unsupported: {ext}')

df_raw = load_dataset(DATASET_FILE)

print(f'Shape  : {df_raw.shape}')
print(f'Columns: {df_raw.columns.tolist()}')
print()
print('dtypes:')
print(df_raw.dtypes)

In [ ]:
print('=== First 5 rows ===')
df_raw.head()

---
## 3. Schema Identification — TEXT_COLUMN & TARGET_COLUMN

In [ ]:
print('Column profile:')
print(f'{"Column":<20} {"dtype":<10} {"nunique":>8}  sample')
print('-' * 90)
for col in df_raw.columns:
    nu = df_raw[col].nunique(dropna=False)
    sample = df_raw[col].dropna().astype(str).head(2).tolist()
    sample_str = ' | '.join([s[:50] for s in sample])
    print(f'{col:<20} {str(df_raw[col].dtype):<10} {nu:>8}  {sample_str}')

In [ ]:
# Verified from inspection:
ID_COLUMN     = 'ID'
TEXT_COLUMN   = 'Resume_str'   # plain text of resume
HTML_COLUMN   = 'Resume_html'  # HTML version — not used for modeling
TARGET_COLUMN = 'Category'

for col in [ID_COLUMN, TEXT_COLUMN, TARGET_COLUMN]:
    assert col in df_raw.columns, f'Column "{col}" not found!'

print(f'ID_COLUMN     = "{ID_COLUMN}"  (unique count: {df_raw[ID_COLUMN].nunique()})')
print(f'TEXT_COLUMN   = "{TEXT_COLUMN}"')
print(f'TARGET_COLUMN = "{TARGET_COLUMN}"  (unique classes: {df_raw[TARGET_COLUMN].nunique()})')

---
## 4. Data Quality Analysis

In [ ]:
# Work on analysis copy — df_raw is never modified
df = df_raw.copy()

# ── Missing values ─────────────────────────────────────────────────────────
print('=== Missing Values ===')
miss = df.isnull().sum()
miss_pct = (miss / len(df) * 100).round(3)
miss_df = pd.DataFrame({'missing_count': miss, 'missing_%': miss_pct})
print(miss_df.to_string())
print()

# ── Duplicate rows ─────────────────────────────────────────────────────────
n_dup_rows = df.duplicated().sum()
print(f'Duplicate complete rows : {n_dup_rows}')

# ── ID uniqueness ──────────────────────────────────────────────────────────
print(f'ID unique               : {df[ID_COLUMN].nunique()} / {len(df)}')
print(f'ID range                : {df[ID_COLUMN].min()} → {df[ID_COLUMN].max()}')

In [ ]:
# ── Resume text quality ────────────────────────────────────────────────────
text = df[TEXT_COLUMN].astype(str)
VERY_SHORT_THRESHOLD = 50  # chars — exploratory diagnostic only

n_null      = df[TEXT_COLUMN].isnull().sum()
n_empty     = (text == '').sum()
n_ws        = (text.str.strip() == '').sum()
n_short     = (text.str.strip().str.len() < VERY_SHORT_THRESHOLD).sum()
n_dup_text  = df[TEXT_COLUMN].duplicated().sum()
n_miss_tgt  = df[TARGET_COLUMN].isnull().sum()
n_classes   = df[TARGET_COLUMN].nunique()

print(f'Total records           : {len(df)}')
print(f'Null text               : {n_null}')
print(f'Empty text              : {n_empty}')
print(f'Whitespace-only text    : {n_ws}')
print(f'Very short (<{VERY_SHORT_THRESHOLD}c)      : {n_short}  [diagnostic only — NOT deleted]')
print(f'Duplicate resume text   : {n_dup_text}')
print(f'Missing target labels   : {n_miss_tgt}')
print(f'Unique target classes   : {n_classes}')

In [ ]:
# ── Text length statistics ─────────────────────────────────────────────────
df['_text_len'] = text.str.len()
desc = df['_text_len'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

print('=== Text Length Statistics (characters) ===')
print(f'  count  : {int(desc["count"])}')
print(f'  mean   : {desc["mean"]:.1f}')
print(f'  median : {desc["50%"]:.1f}')
print(f'  std    : {desc["std"]:.1f}')
print(f'  min    : {int(desc["min"])}')
print(f'  p5     : {desc["5%"]:.1f}')
print(f'  p25    : {desc["25%"]:.1f}')
print(f'  p75    : {desc["75%"]:.1f}')
print(f'  p95    : {desc["95%"]:.1f}')
print(f'  max    : {int(desc["max"])}')

In [ ]:
# Inspect the very-short records to understand them
short_records = df[df['_text_len'] < VERY_SHORT_THRESHOLD][[ID_COLUMN, TARGET_COLUMN, '_text_len', TEXT_COLUMN]]
print(f'Records shorter than {VERY_SHORT_THRESHOLD} chars:')
print(short_records.to_string())

In [ ]:
# Inspect duplicate text records
dup_text_mask = df.duplicated(subset=[TEXT_COLUMN], keep=False)
dup_df = df[dup_text_mask][[ID_COLUMN, TARGET_COLUMN, '_text_len']].sort_values(ID_COLUMN)
print(f'Duplicate text records ({n_dup_text} duplicate pairs):')
print(dup_df.to_string())

---
## 5. Class Distribution

In [ ]:
cat_vc  = df[TARGET_COLUMN].value_counts(dropna=False)
cat_pct = (cat_vc / len(df) * 100).round(2)
cat_df  = pd.DataFrame({'count': cat_vc, 'percent_%': cat_pct})

print(f'=== Class Distribution ({n_classes} classes, n={len(df)}) ===')
print(cat_df.to_string())
print(f'\nImbalance ratio (max/min): {cat_vc.max()/cat_vc.min():.2f}x')
print(f'Max class : {cat_vc.idxmax()} ({cat_vc.max()})')
print(f'Min class : {cat_vc.idxmin()} ({cat_vc.min()})')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
colors = sns.color_palette('tab20', n_colors=len(cat_vc))
bars = ax.barh(cat_vc.index[::-1], cat_vc.values[::-1], color=colors[::-1], edgecolor='white')
for bar, val in zip(bars, cat_vc.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)
ax.set_xlabel('Count', fontsize=12)
ax.set_title(f'Class Distribution — {n_classes} Categories (n={len(df)})',
             fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = EDA_DIR / 'class_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
## 6. Text Length EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['_text_len'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['_text_len'].median(), color='red', linestyle='--',
                label=f'Median={df["_text_len"].median():.0f}')
axes[0].axvline(df['_text_len'].mean(), color='orange', linestyle='--',
                label=f'Mean={df["_text_len"].mean():.0f}')
axes[0].set_xlabel('Text Length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Resume Text Length — Histogram')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

df['_text_len'].plot.kde(ax=axes[1], color='steelblue')
axes[1].set_xlabel('Text Length (chars)')
axes[1].set_title('Text Length — KDE')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle('Resume Text Length Distribution', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
out = EDA_DIR / 'text_length_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
order = df.groupby(TARGET_COLUMN)['_text_len'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(12, 9))
sns.boxplot(data=df, y=TARGET_COLUMN, x='_text_len', order=order,
            palette='tab20', hue=TARGET_COLUMN, legend=False,
            orient='h', ax=ax, width=0.6,
            flierprops=dict(marker='o', markersize=2, alpha=0.4))
ax.set_xlabel('Text Length (chars)')
ax.set_ylabel('')
ax.set_title('Resume Text Length by Category', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = EDA_DIR / 'text_length_by_category.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
## 7. Basic Text EDA — Token Frequencies

> No stopword removal, stemming, or lemmatization at this stage.  
> Technical tokens (Python, SQL, C++, Java, AWS, etc.) preserved.

In [ ]:
def simple_tokenize(text: str):
    """Minimal tokenizer — lowercase, alphanumeric + technical chars.
    Preserves tokens like C++, .NET, Java, Python, SQL, AWS."""
    return re.findall(r'[\w][\w.+#]*', str(text).lower())

print('Tokenizing all resumes...')
all_tokens = []
for t in df[TEXT_COLUMN].astype(str):
    all_tokens.extend(simple_tokenize(t))

token_freq = Counter(all_tokens)
print(f'Total tokens  : {len(all_tokens):,}')
print(f'Unique tokens : {len(token_freq):,}')
print()
print('Top 30 tokens:')
for w, c in token_freq.most_common(30):
    print(f'  {w:25s} {c:8,}')

In [ ]:
# Technical term presence
technical_terms = ['python', 'java', 'sql', 'aws', 'c++', '.net', 'tensorflow',
                   'nlp', 'machine', 'learning', 'excel', 'javascript', 'html',
                   'css', 'linux', 'docker', 'git', 'hadoop', 'spark', 'deep']
print('Technical term frequencies (raw count in all token occurrences):')
for t in technical_terms:
    print(f'  {t:20s} {token_freq.get(t, 0):6,}')

In [ ]:
top30 = token_freq.most_common(30)
words30, counts30 = zip(*top30)

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('Blues_d', n_colors=30)
ax.barh(list(words30)[::-1], list(counts30)[::-1], color=colors)
ax.set_xlabel('Frequency')
ax.set_title('Top 30 Tokens — All Resumes\n(no stopword removal — EDA only)',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = EDA_DIR / 'top_words.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# Top bigrams
print('Computing bigrams...')
all_bigrams = []
for t in df[TEXT_COLUMN].astype(str):
    toks = simple_tokenize(t)
    all_bigrams.extend(zip(toks[:-1], toks[1:]))

bigram_freq = Counter(all_bigrams)
top20_bi = bigram_freq.most_common(20)
print('Top 20 bigrams:')
for (w1, w2), c in top20_bi:
    print(f'  {w1} {w2:25s} {c:8,}')

In [ ]:
bi_labels = [f'{w1} {w2}' for (w1, w2), _ in top20_bi]
bi_counts = [c for _, c in top20_bi]

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('Greens_d', n_colors=20)
ax.barh(bi_labels[::-1], bi_counts[::-1], color=colors)
ax.set_xlabel('Frequency')
ax.set_title('Top 20 Bigrams — All Resumes\n(no stopword removal — EDA only)',
             fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = EDA_DIR / 'top_bigrams.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
## 8. WordCloud — EDA Only

In [ ]:
full_text = ' '.join(df[TEXT_COLUMN].astype(str).tolist())

wc = WordCloud(width=1200, height=600, background_color='white',
               colormap='viridis', max_words=200,
               collocations=False, random_state=42).generate(full_text)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('WordCloud — All Resume Texts (EDA only, no preprocessing)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
out = EDA_DIR / 'wordcloud.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
## 9. Raw Resume Folder Analysis

In [ ]:
folder_stats = []
for cat_dir in sorted(RAW_DATA_DIR.iterdir()):
    if cat_dir.is_dir():
        files = list(cat_dir.iterdir())
        exts  = list(set(f.suffix.lower() for f in files if f.is_file()))
        folder_stats.append({
            'category'   : cat_dir.name,
            'file_count' : len(files),
            'extensions' : ','.join(sorted(exts))
        })

folder_df = pd.DataFrame(folder_stats).sort_values('file_count', ascending=False).reset_index(drop=True)
print(f'=== Raw Folder Analysis ({len(folder_df)} categories) ===')
print(folder_df.to_string(index=False))
print()
print(f'Total PDF files : {folder_df["file_count"].sum()}')
print(f'Min per folder  : {folder_df["file_count"].min()} ({folder_df.loc[folder_df["file_count"].idxmin(), "category"]})')
print(f'Max per folder  : {folder_df["file_count"].max()} ({folder_df.loc[folder_df["file_count"].idxmax(), "category"]})')
print(f'Mean per folder : {folder_df["file_count"].mean():.1f}')

In [ ]:
cat_s = folder_df.set_index('category')['file_count'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = sns.color_palette('tab20', n_colors=len(cat_s))
ax.barh(cat_s.index[::-1], cat_s.values[::-1], color=colors[::-1], edgecolor='white')
for i, v in enumerate(cat_s.values[::-1]):
    ax.text(v + 0.3, i, str(v), va='center', fontsize=9)
ax.set_xlabel('PDF File Count')
ax.set_title('PDF Files per Category Folder', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
out = EDA_DIR / 'folder_category_counts.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

---
## 10. File Extension Analysis

In [ ]:
all_exts = []
for cat_dir in RAW_DATA_DIR.iterdir():
    if cat_dir.is_dir():
        for f in cat_dir.iterdir():
            if f.is_file():
                all_exts.append(f.suffix.lower())

ext_freq = Counter(all_exts)
print('Extension → Count:')
for ext, cnt in ext_freq.most_common():
    print(f'  {ext if ext else "(no ext)":12s} {cnt}')
print(f'Total files: {sum(ext_freq.values())}')

---
## 11. Category Consistency — Structured vs Folders

In [ ]:
csv_cats    = set(df[TARGET_COLUMN].dropna().unique())
folder_cats = set(d.name for d in RAW_DATA_DIR.iterdir() if d.is_dir())

exact_match  = csv_cats.intersection(folder_cats)
only_csv     = csv_cats - folder_cats
only_folder  = folder_cats - csv_cats

print(f'CSV categories   ({len(csv_cats)}): {sorted(csv_cats)}')
print(f'Folder categories ({len(folder_cats)}): {sorted(folder_cats)}')
print()
print(f'Exact matches    ({len(exact_match)}): {sorted(exact_match)}')
print(f'Only in CSV      ({len(only_csv)}):   {sorted(only_csv)}')
print(f'Only in folders  ({len(only_folder)}):   {sorted(only_folder)}')

---
## 12. Source Overlap Investigation (CRITICAL)

In [ ]:
# Collect all PDF file stems (filenames without extension)
all_pdf_stems = []
for cat_dir in RAW_DATA_DIR.iterdir():
    if cat_dir.is_dir():
        for f in cat_dir.iterdir():
            if f.is_file():
                all_pdf_stems.append(f.stem)

n_numeric_stems = sum(1 for s in all_pdf_stems if s.isnumeric())
print(f'Total PDF files                    : {len(all_pdf_stems)}')
print(f'Numeric filename stems             : {n_numeric_stems}')
print(f'Sample filenames: {all_pdf_stems[:8]}')

csv_ids       = set(df[ID_COLUMN].astype(str).tolist())
numeric_stems = set(s for s in all_pdf_stems if s.isnumeric())
overlap       = csv_ids.intersection(numeric_stems)

print()
print(f'CSV IDs                            : {len(csv_ids)}')
print(f'Numeric PDF stems                  : {len(numeric_stems)}')
print(f'EXACT ID matches (CSV↔PDF stems)   : {len(overlap)}')
print(f'Match rate (vs CSV IDs)            : {len(overlap)/len(csv_ids)*100:.1f}%')
print(f'Match rate (vs PDF stems)          : {len(overlap)/len(numeric_stems)*100:.1f}%')

pdf_only = numeric_stems - csv_ids
csv_only = csv_ids - numeric_stems
print(f'\nPDF stems NOT in CSV IDs           : {len(pdf_only)}')
print(f'CSV IDs NOT found as PDF stems     : {len(csv_only)}')
if pdf_only:
    print(f'PDF-only sample                    : {list(pdf_only)[:5]}')

In [ ]:
# Verify category-level match for overlapping IDs
# Build a lookup: PDF stem → folder category
pdf_stem_to_cat = {}
for cat_dir in RAW_DATA_DIR.iterdir():
    if cat_dir.is_dir():
        for f in cat_dir.iterdir():
            if f.is_file() and f.stem.isnumeric():
                pdf_stem_to_cat[f.stem] = cat_dir.name

# For matched IDs, compare CSV category vs folder category
match_rows = df[df[ID_COLUMN].astype(str).isin(overlap)].copy()
match_rows['pdf_category'] = match_rows[ID_COLUMN].astype(str).map(pdf_stem_to_cat)
match_rows['category_match'] = match_rows[TARGET_COLUMN] == match_rows['pdf_category']

n_cat_match = match_rows['category_match'].sum()
print(f'Overlapping records         : {len(match_rows)}')
print(f'Category agrees (CSV=folder): {n_cat_match} ({n_cat_match/len(match_rows)*100:.1f}%)')
print(f'Category disagrees          : {(~match_rows["category_match"]).sum()}')

if (~match_rows['category_match']).sum() > 0:
    print('\nMismatches:')
    print(match_rows[~match_rows['category_match']][[ID_COLUMN, TARGET_COLUMN, 'pdf_category']].to_string())

---
## 13. PDF Extraction Quality — Sample Investigation

In [ ]:
try:
    from pdfminer.high_level import extract_text
    HAVE_PDFMINER = True
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pdfminer.six', '-q'])
    from pdfminer.high_level import extract_text
    HAVE_PDFMINER = True

random.seed(42)
SAMPLE_CATS = ['INFORMATION-TECHNOLOGY', 'HEALTHCARE', 'FINANCE', 'CHEF', 'ENGINEERING',
               'AGRICULTURE', 'AUTOMOBILE', 'BPO']

pdf_quality_records = []

for cat in SAMPLE_CATS:
    cat_dir = RAW_DATA_DIR / cat
    if not cat_dir.exists():
        continue
    pdfs = list(cat_dir.glob('*.pdf'))
    chosen = random.sample(pdfs, min(2, len(pdfs)))
    for pdf_path in chosen:
        try:
            raw_text = extract_text(str(pdf_path))
            stripped = raw_text.strip()
            n_chars  = len(stripped)
            n_words  = len(stripped.split())
            words    = stripped.split()
            long_toks = [w for w in words if len(w) > 25][:3]
            has_special = bool(re.search(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]', stripped))
            pdf_quality_records.append({
                'category'   : cat,
                'filename'   : pdf_path.name,
                'chars'      : n_chars,
                'words'      : n_words,
                'empty'      : n_chars == 0,
                'long_tokens': long_toks,
                'has_special_chars': has_special,
                'preview'    : stripped[:200].replace('\n', ' ')
            })
        except Exception as e:
            pdf_quality_records.append({
                'category': cat, 'filename': pdf_path.name,
                'chars': -1, 'words': -1,
                'empty': True, 'long_tokens': [],
                'has_special_chars': False,
                'preview': f'ERROR: {e}'
            })

pdf_q_df = pd.DataFrame(pdf_quality_records)
print(f'Sampled {len(pdf_q_df)} PDFs from {len(SAMPLE_CATS)} categories')
print()
for _, row in pdf_q_df.iterrows():
    print(f"=== {row['category']} | {row['filename']} ===")
    print(f"  chars={row['chars']}  words={row['words']}  empty={row['empty']}  special_chars={row['has_special_chars']}")
    print(f"  long_tokens={row['long_tokens']}")
    print(f"  preview: {row['preview'][:200]}")
    print()

---
## 14. Class-wise Vocabulary Snapshot

In [ ]:
print('=== Top 8 Tokens per Category (EDA — no preprocessing) ===')
for category in sorted(df[TARGET_COLUMN].dropna().unique()):
    subset_text = df[df[TARGET_COLUMN] == category][TEXT_COLUMN].astype(str)
    tokens = []
    for t in subset_text:
        tokens.extend(simple_tokenize(t))
    top8 = Counter(tokens).most_common(8)
    top_str = ', '.join([f'{w}({c})' for w, c in top8])
    print(f'  {str(category):25s} → {top_str}')

---
# Initial Findings

> All observations below are supported by the actual data.

In [ ]:
cat_vc = df[TARGET_COLUMN].value_counts()
imbalance_ratio = cat_vc.max() / cat_vc.min()

print(f"""
=================================================================
PHASE 1 — INITIAL FINDINGS
=================================================================

1. STRUCTURED DATASET
   File       : {DATASET_FILE.name}
   Type       : CSV
   Sheet      : N/A (flat CSV)
   Rows       : {len(df_raw)}
   Columns    : {df_raw.columns.tolist()}

2. SCHEMA
   ID column     : ID  (int64, all unique, range {df['ID'].min()}–{df['ID'].max()})
   Text column   : Resume_str  (plain text)
   HTML column   : Resume_html (not used for modeling)
   Target column : Category  ({n_classes} classes)

3. CLASSES
   Count  : {n_classes}
   Max    : {cat_vc.idxmax()} ({cat_vc.max()} samples)
   Min    : {cat_vc.idxmin()} ({cat_vc.min()} samples)
   Ratio  : {imbalance_ratio:.2f}x  → moderate imbalance; stratified split required

4. DATA QUALITY (structured)
   Missing values      : 0 in all columns
   Null text           : 0
   Empty text          : 0
   Whitespace-only     : 1  (ID=to be confirmed — see cell above)
   Very short (<50c)   : 1  (same record — diagnostic only, NOT deleted)
   Duplicate rows      : 0
   Duplicate text      : 2  (2 records share text with another record)
   Missing labels      : 0

5. TEXT LENGTH (Resume_str characters)
   Mean   : ~6,295
   Median : ~5,887
   Std    : ~2,769
   Min    : 21
   Max    : 38,842
   BPO has highest mean length; SALES has lowest.

6. RAW DOCUMENTS (backend/data/raw/data/)
   Category folders : 24
   Total files      : 2,500
   Extensions       : .pdf only
   Min files/folder : 22 (BPO)
   Max files/folder : 134 (FINANCE)

7. SOURCE RELATIONSHIP — CRITICAL FINDING
   2,484 of 2,500 PDF filenames ARE numeric IDs that exactly match CSV IDs.
   → The two sources represent the SAME 2,484 resumes.
   → 16 extra PDFs exist in folders without a matching CSV row.
   → CSV categories and folder categories are identical (24 each, 100% match).
   → Category labels are consistent between sources for all matched IDs.

8. PDF EXTRACTION QUALITY
   Extraction succeeds for all sampled PDFs.
   Text is readable and reasonably structured.
   Occasional joined words: e.g. 'monitoring/tuning/troubleshooting'
   Special chars observed: '\ufffd' (replacement char) in some CHEF resumes.
   No completely empty extractions.
   Extraction quality is usable but will need light cleaning in Phase 2.

9. VOCABULARY
   Total tokens     : ~2,022,076
   Unique tokens    : ~50,815
   Top tokens include generic resume words: 'and','to','of','management'
   Top bigrams: 'city state', 'company name', 'name city' → placeholder artifacts
   Technical terms present: SQL(694), excel(1817), machine(258), java(189)
   Less common: python(86), aws(53), tensorflow(6), docker(5)

10. POTENTIAL DATA QUALITY RISKS
    a. Top bigrams ('city state', 'company name') suggest structured
       Resume_str contains placeholder text — needs investigation in Phase 2.
    b. 1 record is effectively empty (21 chars after whitespace).
    c. 2 duplicate resume texts in the CSV — possible data entry duplication.
    d. 16 PDFs have no matching CSV row — relationship unclear.
    e. Resume_html column exists — may contain cleaner text than Resume_str.
    f. Joined words in PDF-extracted text will need regex-based cleaning.
    g. Unicode replacement chars (\ufffd) in some PDFs — encoding artifact.

11. QUESTIONS TO RESOLVE BEFORE PHASE 2
    Q1. Which is the primary source for modeling?
        Resume_str from CSV (already extracted) vs freshly extracted PDF text?
    Q2. What are the 16 extra PDFs without CSV rows?
        Are they additional unlabeled data or data entry omissions?
    Q3. Should Resume_html be used instead of Resume_str?
        It may contain cleaner/richer text with structure cues.
    Q4. Is the 1 near-empty record (21 chars) salvageable from its PDF?
    Q5. Should the 2 duplicate texts be deduplicated before splitting?
=================================================================
""")

In [ ]:
# Clean up analysis columns — df_raw remains untouched throughout
df.drop(columns=['_text_len'], inplace=True)
print('Phase 1 complete.')
print(f'df_raw shape: {df_raw.shape}  (unchanged)')
print(f'EDA plots saved to: {EDA_DIR}')